# Stage 5 · Validation error analysis

Review the recorded YOLO26s outputs from the frozen validation split. All figures and findings are read from the Stage 5 archive.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from roadguard.utils.paths import project_root

root = project_root()
selected = os.environ.get("ROADGUARD_STAGE5_RUN")
if selected:
    run = Path(selected)
    result_path = run / "stage5_result.json"
else:
    result_path = root / "artifacts/experiments/stage5_result.json"
    run = root / json.loads(result_path.read_text(encoding="utf-8"))["run"]
result = json.loads(result_path.read_text(encoding="utf-8"))
metadata = json.loads((run / "metadata.json").read_text(encoding="utf-8"))
print(run.name)

## Checkpoint and full-validation coverage

The archived Ultralytics metrics use its validation operating point. The diagnostic counts use confidence ≥ 0.25 and one-to-one, same-class matching at IoU ≥ 0.50.

In [ ]:
print(f"{result['validation_images']:,} images, {result['evaluated_annotations']:,} annotations")
print(f"Checkpoint epoch: {metadata['checkpoint_epoch']}")
print(f"TP: {result['true_positives']:,}")
print(f"FP: {result['false_positives']:,} · FN: {result['false_negatives']:,}")
print("Checkpoint SHA-256:", result["integrity"]["checkpoint_sha256"])
metrics = ["precision", "recall", "map50", "map50_95"]
display(
    pd.DataFrame(
        {
            "Stage 4": metadata["stage4_validation_metrics"],
            "Stage 5": result["validation_metrics"],
        },
        index=metrics,
    ).round(4)
)

## Per-class errors and confusion

Matrix rows are ground-truth categories. The background column counts defects without a paired detection at IoU ≥ 0.50. The background row counts unpaired detections.

In [ ]:
columns = [
    "class",
    "evaluated_annotations",
    "precision",
    "recall",
    "ap50",
    "ap50_95",
    "false_positives",
    "false_negatives",
    "false_negative_rate",
]
display(pd.read_csv(run / "per_class_metrics.csv")[columns].round(4))
for name in ("per_class_performance.png", "confusion_matrix.png"):
    display(Image(filename=str(run / "figures" / name), width=940))
display(pd.DataFrame(result["confusion_pairs"]).head(5).round(4))

## Object size and confidence

Size boundaries are the 33⅓ and 66⅔ percentiles of relative validation annotation area. The score figure includes predictions from the existing 0.001 evaluation floor. The operating threshold stays at 0.25.

In [ ]:
definition = result["size_definition"]
for key in ("small_max_inclusive", "medium_max_inclusive"):
    print(f"{key}: {definition[key]:.3%} of image area")
for name in ("object_size.png", "confidence_distribution.png"):
    display(Image(filename=str(run / "figures" / name), width=780))
display(pd.DataFrame(result["confidence_analysis"]).T.round(4))

## Representative errors and hard scenes

The gallery ranks examples by severity with class and image coverage. In each pair, the left panel contains ground truth and the right panel contains model outputs at the operating threshold. The seeded crop audit separately records visible cues with uncertainty retained.

In [ ]:
gallery = json.loads((run / "gallery_index.json").read_text(encoding="utf-8"))
for category in ("false_positives", "false_negatives", "localization_errors", "class_confusions"):
    print(category.replace("_", " ").title())
    display(Image(filename=str(run / gallery[category][0]["figure"]), width=960))
display(pd.read_csv(run / "hard_examples.csv").head(5))
display(pd.DataFrame(result["visual_audit"]["counts"]).fillna(0).astype(int))

## Stage decision

The validation findings are suitable for documenting limitations before a separately approved final test evaluation.

In [ ]:
print("Strongest by AP50–95:", result["strongest_ap_class"])
print("Weakest by AP50–95:", result["weakest_ap_class"])
print("Highest miss rate:", result["highest_miss_rate_class"])
print(f"Misses without a paired detection: {result['annotations_to_background']:,}")
print("Validation checksums match:", result["integrity"]["validation_checksums_match"])
print("Test data evaluated:", result["integrity"]["test_evaluated"])
print("Candidate:", result["checkpoint"])